In [1]:
import pandas as pd
import numpy as np

from scipy.stats import (
    ttest_ind,
    f_oneway,
    chi2_contingency
)

In [4]:
df = pd.read_csv("../data/cleaned_data.csv")

/var/folders/td/g0nj9lw14y165dmqgtz9jxzh0000gn/T/ipykernel_7504/2527705676.py:1: DtypeWarning: Columns (0: CapitalOutstanding, 1: CrossBorder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/cleaned_data.csv")


Feature Engineering

In [5]:
df["LossRatio"] = np.where(
    df["TotalPremium"] > 0,
    df["TotalClaims"] / df["TotalPremium"],
    np.nan
)

df["Margin"] = (
    df["TotalPremium"] - df["TotalClaims"]
)

df["LossRatio"] = df["LossRatio"].replace(
    [np.inf, -np.inf],
    np.nan
)

Province Risk Difference

In [6]:
province_groups = [
    group["LossRatio"].dropna()
    for _, group in df.groupby("Province")
]

anova_result = f_oneway(*province_groups)

print("Province Risk Difference")
print("F-statistic:", anova_result.statistic)
print("p-value:", anova_result.pvalue)

Province Risk Difference
F-statistic: 4.96918412883474
p-value: 3.562505015220851e-06


Gender Claim Difference

In [9]:
male_claims = df[
    df["Gender"] == "Male"
]["TotalClaims"]

female_claims = df[
    df["Gender"] == "Female"
]["TotalClaims"]

gender_test = ttest_ind(
    male_claims,
    female_claims,
    nan_policy="omit"
)

print("Gender Claim Difference")
print("t-statistic:", gender_test.statistic)
print("p-value:", gender_test.pvalue)

Gender Claim Difference
t-statistic: -0.24803623812388725
p-value: 0.8041073961270342


Postal Code Risk Difference

In [10]:
postal_groups = [
    group["LossRatio"].dropna()
    for _, group in df.groupby("PostalCode")
]

postal_anova = f_oneway(*postal_groups[:20])

print("Postal Code Risk Difference")
print("F-statistic:", postal_anova.statistic)
print("p-value:", postal_anova.pvalue)

Postal Code Risk Difference
F-statistic: 0.9954750736605757
p-value: 0.46250326407216846


In [11]:
vehicle_groups = [
    group["Margin"].dropna()
    for _, group in df.groupby("VehicleType")
]

vehicle_test = f_oneway(*vehicle_groups)

print("Vehicle Margin Difference")
print("F-statistic:", vehicle_test.statistic)
print("p-value:", vehicle_test.pvalue)

Vehicle Margin Difference
F-statistic: 1.1343963860666528
p-value: 0.33811471779480706


Final Summary Table

In [12]:
summary = pd.DataFrame({
    "Hypothesis": [
        "Province Risk Difference",
        "Gender Claim Difference",
        "Postal Code Risk Difference",
        "Vehicle Margin Difference"
    ],
    "Test": [
        "ANOVA",
        "Independent t-test",
        "ANOVA",
        "ANOVA"
    ],
    "p-value": [
        anova_result.pvalue,
        gender_test.pvalue,
        postal_anova.pvalue,
        vehicle_test.pvalue
    ]
})

summary["Reject Null?"] = (
    summary["p-value"] < 0.05
)

summary

,Hypothesis,Test,p-value,Reject Null?
0,Province Risk Difference,ANOVA,0.000004,True
1,Gender Claim Difference,Independent t-test,0.804107,False
2,Postal Code Risk Difference,ANOVA,0.462503,False
3,Vehicle Margin Difference,ANOVA,0.338115,False
